# JANORA — NLLB translation evaluation

This notebook checks the translation module on the `translation/nllb-migration` branch. In Colab, select **Runtime → Change runtime type → T4 GPU** when available. CPU also works, but NLLB inference will be slower.

The first NLLB run downloads model weights (about 2.5 GB).

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!rm -rf marg
!git clone --depth 1 --branch translation/nllb-migration https://github.com/YogeshBijalwan2211/marg.git
%cd marg
!git status --short --branch

In [ ]:
# NLLB dependencies; no IndicTransToolkit or Hugging Face login is required
!pip install -q -r requirements-translation.txt

In [ ]:
# Model-free regression tests; these should not download NLLB weights
!python -m pytest tests/test_translation.py -q

In [ ]:
# Offline checks: language ID heuristic, number normalisation, glossary masking
!python src/eval_translation.py offline --split val --heuristic-langid

In [ ]:
# End-to-end NLLB evaluation, with IndicXlit enabled for romanised Indic inputs
!python src/eval_translation.py mt --split val --backend nllb --limit 300

In [ ]:
# Optional ablation: compare without roman-to-native transliteration
!python src/eval_translation.py mt --split val --backend nllb --no-translit --limit 300

## What to inspect

- The pytest cell should report all tests passing.
- The offline report shows language-ID, age/income normalisation, and scheme glossary coverage.
- The MT report measures whether key profile facts survive translation, by language, and reports the translation gate-failure rate.
- Compare the final two MT cells to see the effect of transliteration on Latin-script Indic inputs.

The notebook uses the validation split. Reserve the test split for the final report.